# Binary fraction across metallicity

In [ ]:
from posydon.utils.common_functions import convert_metallicity_to_string, orbital_separation_from_period
from posydon.popsyn.distributions import PowerLawPeriod
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from cmap import Colormap
import pickle
from pathlib import Path
from tqdm import tqdm
from scipy.stats import gaussian_kde
from helper_functions import calc_Teff, optical_depth_calculation

tol_vibrant = Colormap('tol:vibrant')(np.linspace(0, 1, 7))

COLORS = {0.0001: '#1f4e79',
           0.001: '#2e75b6',
            0.01: '#5ba3d0', 
             0.1: '#f4a261',
             0.2: '#e76f51',
            0.45: '#c0392b', 
             1.0: '#8e1a0e',
             2.0: '#5c0a0a'
             }

In [ ]:
def load_metallicity(metallicity, data_folder):
    """Load and process WR phase data for a single metallicity.

    Returns a dict of arrays ready for analysis.
    """
    met_str = convert_metallicity_to_string(metallicity)
    file_path = data_folder / f'WR_phases_Z{met_str}_Zsun.pkl'
    with open(file_path, 'rb') as f:
        data = pickle.load(f)

    S1_WR_data = data['S1_WR_data']
    S2_WR_data = data['S2_WR_data']

    s1 = pd.concat(S1_WR_data.values(), keys=S1_WR_data.keys())
    s2 = pd.concat(S2_WR_data.values(), keys=S2_WR_data.keys())

    s1_Teff = 10**calc_Teff(s1['S1_log_L'], s1['S1_log_R'])
    s1_sep  = orbital_separation_from_period(s1['orbital_period'].values,
                                              s1['S1_mass'].values,
                                              s1['S2_mass'].values)
    s2_Teff = 10**calc_Teff(s2['S2_log_L'], s2['S2_log_R'])
    s2_sep  = orbital_separation_from_period(s2['orbital_period'].values,
                                              s2['S2_mass'].values,
                                              s2['S1_mass'].values)

    return {
        'time':             np.concatenate([s1['time'].values,               s2['time'].values]),
        'weight':           np.concatenate([s1['weight'].values,             s2['weight'].values]),
        'state':            np.concatenate([s1['state'].values,              s2['state'].values]),
        'key':              np.concatenate([s1.index.get_level_values(0),    s2.index.get_level_values(0)]),
        'star':             np.array(['S1'] * len(s1) + ['S2'] * len(s2)),
        'period':           np.concatenate([s1['orbital_period'].values,     s2['orbital_period'].values]),
        'separation':       np.concatenate([s1_sep,                          s2_sep]),
        'Teff':             np.concatenate([s1_Teff.values,                  s2_Teff.values]),
        'L':                np.concatenate([10**s1['S1_log_L'].values,       10**s2['S2_log_L'].values]),
        'optical_depth':    np.concatenate([s1['optical_depth'].values,      s2['optical_depth'].values]),
        'surface_H':        np.concatenate([s1['S1_surface_h1'].values,      s2['S2_surface_h1'].values]),
        'surface_He':       np.concatenate([s1['S1_surface_he4'].values,     s2['S2_surface_he4'].values]),
        'mass':             np.concatenate([s1['S1_mass'].values,            s2['S2_mass'].values]),
        'mass_companion':   np.concatenate([s1['S2_mass'].values,            s2['S1_mass'].values]),
        'radius':           np.concatenate([10**s1['S1_log_R'].values,       10**s2['S2_log_R'].values]),
        'Mdot':             np.concatenate([10**s1['S1_lg_mdot'].values,     10**s2['S2_lg_mdot'].values]),
        'companion':        np.concatenate([s1['S2_state'].values,           s2['S1_state'].values]),
        'S1_state':         np.concatenate([s1['S1_state'].values,           s2['S2_state'].values]),
        'model_weight':     np.concatenate([s1['model_weight'].values,       s2['model_weight'].values]),
    }


data_folder = Path('/Users/max/Documents/projects/WR_population/data/optical_depth/')
metallicities = [0.01, 0.1, 0.2, 0.45, 1.0]#[0.01, 0.1, 0.2, 0.45, 1.0]  # in units of Zsun

WR_data = {Z: load_metallicity(Z, data_folder) for Z in metallicities}

In [ ]:
def classify_WR(d):
    """Return WNL, WNE, WC boolean masks for a data dict from load_metallicity."""
    WNL = (
        (d['surface_H'] > 0.05) & (d['surface_H'] <= 0.5)
        & (d['optical_depth'] > 1.5)
        & ~pd.Series(d['S1_state']).str.contains('H_burning', na=False).values
    )
    WNE = (d['surface_H'] <= 0.05) & (d['surface_He'] >= 0.7) & (d['optical_depth'] > 1.5)
    WC  = (d['surface_H'] <= 0.05) & (d['surface_He'] <  0.7) & (d['optical_depth'] > 1.5)
    return WNL, WNE, WC


def make_binary_mask(d):
    """Return mask for detached/RLO/contact binaries."""
    s = d['state']
    return (s == 'detached') | (s == 'RLO1') | (s == 'contact') | (s == 'RLO2')


In [ ]:
# Example: compute WR type fractions for each metallicity
for Z, d in WR_data.items():
    WNL, WNE, WC = classify_WR(d)
    bm = make_binary_mask(d)
    w = d['weight'] * d['model_weight']
    total = np.sum(w[bm & (WNL | WNE | WC)])
    WNL_fraction = np.sum(w[bm & WNL]) / total
    WNE_fraction = np.sum(w[bm & WNE]) / total
    WC_fraction  = np.sum(w[bm & WC])  / total
    print(f"Z = {Z} Zsun  ->  WNL: {WNL_fraction*100:.0f}%  WNE: {WNE_fraction*100:.0f}%  WC: {WC_fraction*100:.0f}%")
    
    # binary fraction
    total_binaries = np.sum(w[bm])
    binary_fraction = total_binaries / np.sum(w)
    print(f"Z = {Z} Zsun  ->  Binary fraction: {binary_fraction*100:.0f}%")
    # per sub-type
    WNL_binaries = np.sum(w[bm & WNL])
    WNE_binaries = np.sum(w[bm & WNE])
    WC_binaries  = np.sum(w[bm & WC])
    print(f"Z = {Z} Zsun  ->  WNL binaries: {(WNL_binaries/np.sum(w[WNL]))*100:.0f}%  WNE binaries: {(WNE_binaries/np.sum(w[WNE]))*100:.0f}%  WC binaries: {(WC_binaries/np.sum(w[WC]))*100:.0f}%")

